In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

dataset

In [ ]:
df = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
test_df = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")

df.head()

X, y from dataset

In [ ]:
X = df.drop("label", axis=1).values/255.0
y = df["label"].values
X_test = test_df.values/255.0

print("X shape: ", X.shape)
print("y shape: ", y.shape)
print("X_test shape: ", X_test.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
X_train = X_train.reshape(-1, 784)
y_train = tf.keras.utils.to_categorical(y_train, 10)
X_val = X_val.reshape(-1, 784)
y_val = tf.keras.utils.to_categorical(y_val, 10)
X_test = X_test.reshape(-1, 784)

print("X_train new shape: ", X_train.shape)
print("y_train new shape: ", y_train.shape)
print("X_val new shape: ", X_val.shape)
print("y_val new shape: ", y_val.shape)
print("X_test new shape: ", X_test.shape)

Model Layers

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

Compiling

In [ ]:
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

Train Model

In [ ]:
history = model.fit(X_train, y_train, epochs=20, batch_size=32)

Cross Validation Accuracy

In [ ]:
loss, accuracy = model.evaluate(X_val, y_val)
print(f"y_val Accuracy: {accuracy:.4f}")

Predictions for test set

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
if 'val_accuracy' in history.history:
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
if 'val_loss' in history.history:
    plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
predictions = model.predict(X_test)
predictions_labels = np.argmax(predictions, axis=1)

Submission CSV

In [ ]:
submission = pd.DataFrame({
    "ImageId": np.arange(1, (len(predictions_labels) + 1)),
    "Label": predictions_labels
})
submission.to_csv("submission.csv", index=False)
submission.head()